# 00 — Data Exploration
## AuctionIQ — IPL Player Valuation & Auction Intelligence
**Goal:** Understand the raw Cricsheet data structure, count matches, seasons, and plan the full pipeline.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Path to raw data
RAW_PATH = Path('../data/raw')

# Count all CSV files
all_files = list(RAW_PATH.glob('*.csv'))
print(f"Total CSV files found: {len(all_files)}")
print(f"\nFirst 5 files:")
for f in all_files[:5]:
    print(f"  {f.name}")

Total CSV files found: 2418

First 5 files:
  1082591.csv
  1082591_info.csv
  1082592.csv
  1082592_info.csv
  1082593.csv


In [3]:
# Load one match file — confirmed working
match_file = all_files[0]
df_match = pd.read_csv(match_file)

print("=== MATCH FILE STRUCTURE ===")
print(f"Shape: {df_match.shape}")
print(f"Columns: {df_match.columns.tolist()}")
print(f"Seasons in this file: {df_match['season'].unique()}")
print(f"Teams: {df_match['batting_team'].unique()}")

# Info file has irregular format — read as raw text instead
info_file = all_files[1]
print("\n=== INFO FILE (raw) ===")
with open(info_file, 'r') as f:
    lines = f.readlines()
for line in lines[:20]:
    print(line.strip())

=== MATCH FILE STRUCTURE ===
Shape: (248, 22)
Columns: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']
Seasons in this file: [2017]
Teams: <StringArray>
['Sunrisers Hyderabad', 'Royal Challengers Bangalore']
Length: 2, dtype: str

=== INFO FILE (raw) ===
version,2.1.0
info,balls_per_over,6
info,team,Sunrisers Hyderabad
info,team,Royal Challengers Bangalore
info,gender,male
info,season,2017
info,date,2017/04/05
info,event,Indian Premier League
info,match_number,1
info,venue,"Rajiv Gandhi International Stadium, Uppal"
info,city,Hyderabad
info,toss_winner,Royal Challengers Bangalore
info,toss_decision,field
info,player_of_match,Yuvraj Singh
info,umpire,AY Dandekar
info,umpire,NJ Llong
info,reserve_umpire,N Pandit
info,tv_umpire,A Deshmukh
info,match_

In [4]:
from tqdm import tqdm

# Separate match files from info files
match_files = [f for f in all_files if '_info' not in f.name]
info_files   = [f for f in all_files if '_info' in f.name]

print(f"Match files : {len(match_files)}")
print(f"Info files  : {len(info_files)}")

# Load a sample of match files to see seasons
seasons = set()
for f in match_files[:50]:
    df = pd.read_csv(f)
    seasons.update(df['season'].unique())

print(f"\nSeasons found (sample): {sorted(seasons)}")

# Check one match file size
sizes = [pd.read_csv(f).shape[0] for f in match_files[:20]]
print(f"\nAvg balls per match: {sum(sizes)//len(sizes)}")
print(f"Min balls: {min(sizes)} | Max balls: {max(sizes)}")

Match files : 1209
Info files  : 1209

Seasons found (sample): [np.int64(2017)]

Avg balls per match: 240
Min balls: 212 | Max balls: 254


In [5]:
# Quick scan all match files for seasons
seasons_all = set()
for f in tqdm(match_files, desc="Scanning seasons"):
    try:
        df = pd.read_csv(f, usecols=['season'])
        seasons_all.update(df['season'].unique())
    except:
        pass

print(f"All seasons found: {sorted(seasons_all)}")
print(f"Total seasons: {len(seasons_all)}")

Scanning seasons: 100%|█████████████████████████████████████████████████████████| 1209/1209 [00:03<00:00, 399.75it/s]


UFuncTypeError: ufunc 'less' did not contain a loop with signature matching types (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.StrDType'>) -> None

In [6]:
# Fix mixed types — convert all seasons to string
seasons_clean = sorted(set(str(s) for s in seasons_all))
print(f"All seasons: {seasons_clean}")
print(f"Total seasons: {len(seasons_clean)}")

All seasons: ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']
Total seasons: 19


In [7]:
# Load all 1209 match files into one master dataframe
dfs = []
failed = 0

for f in tqdm(match_files, desc="Loading matches"):
    try:
        df = pd.read_csv(f)
        dfs.append(df)
    except Exception as e:
        failed += 1

master = pd.concat(dfs, ignore_index=True)

# Standardise season column to string
master['season'] = master['season'].astype(str)

print(f"✅ Master dataset created")
print(f"   Total balls  : {len(master):,}")
print(f"   Total matches: {master['match_id'].nunique():,}")
print(f"   Seasons      : {sorted(master['season'].unique())}")
print(f"   Failed files : {failed}")
print(f"   Columns      : {master.columns.tolist()}")
print(f"\nShape: {master.shape}")

Loading matches: 100%|██████████████████████████████████████████████████████████| 1209/1209 [00:04<00:00, 258.58it/s]


✅ Master dataset created
   Total balls  : 287,513
   Total matches: 1,209
   Seasons      : ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']
   Failed files : 0
   Columns      : ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']

Shape: (287513, 22)


In [8]:
# Extract metadata from all info files
metadata = []

for f in tqdm(info_files, desc="Loading metadata"):
    try:
        match_id = f.name.replace('_info.csv', '')
        match_data = {'match_id': match_id}
        
        with open(f, 'r') as file:
            for line in file:
                line = line.strip()
                parts = line.split(',')
                if len(parts) >= 3 and parts[0] == 'info':
                    key = parts[1].strip()
                    value = ','.join(parts[2:]).strip()
                    # Only keep useful fields
                    if key in ['season', 'date', 'venue', 'city',
                               'winner', 'player_of_match',
                               'toss_winner', 'toss_decision',
                               'event', 'match_number']:
                        match_data[key] = value
        
        metadata.append(match_data)
    except:
        pass

df_meta = pd.DataFrame(metadata)

print(f"✅ Metadata loaded")
print(f"   Matches : {len(df_meta):,}")
print(f"   Columns : {df_meta.columns.tolist()}")
print(f"\nSample:")
print(df_meta.head(3))

Loading metadata: 100%|████████████████████████████████████████████████████████| 1209/1209 [00:00<00:00, 1857.00it/s]

✅ Metadata loaded
   Matches : 1,209
   Columns : ['match_id', 'season', 'date', 'event', 'match_number', 'venue', 'city', 'toss_winner', 'toss_decision', 'player_of_match', 'winner']

Sample:
  match_id season        date                  event match_number  \
0  1082591   2017  2017/04/05  Indian Premier League            1   
1  1082592   2017  2017/04/06  Indian Premier League            2   
2  1082593   2017  2017/04/07  Indian Premier League            3   

                                         venue       city  \
0  "Rajiv Gandhi International Stadium, Uppal"  Hyderabad   
1      Maharashtra Cricket Association Stadium       Pune   
2       Saurashtra Cricket Association Stadium     Rajkot   

                   toss_winner toss_decision player_of_match  \
0  Royal Challengers Bangalore         field    Yuvraj Singh   
1       Rising Pune Supergiant         field       SPD Smith   
2        Kolkata Knight Riders         field         CA Lynn   

                   winner  


In [9]:
# Save master datasets
master.to_csv('../data/processed/master_balls.csv', index=False)
df_meta.to_csv('../data/processed/master_meta.csv', index=False)

print("✅ Datasets saved")
print(f"\n=== AUCTIONIQ DATA SUMMARY ===")
print(f"Ball-by-ball records : {len(master):,}")
print(f"Total matches        : {master['match_id'].nunique():,}")
print(f"Total seasons        : {master['season'].nunique()}")
print(f"Unique batters       : {master['striker'].nunique():,}")
print(f"Unique bowlers       : {master['bowler'].nunique():,}")
print(f"Total runs scored    : {master['runs_off_bat'].sum():,}")
print(f"Total wickets        : {master['wicket_type'].notna().sum():,}")
print(f"Date range           : {master['start_date'].min()} → {master['start_date'].max()}")
print(f"\n📁 Saved:")
print(f"   data/processed/master_balls.csv")
print(f"   data/processed/master_meta.csv")
print(f"\nNext → 01_batting_analysis.ipynb")

✅ Datasets saved

=== AUCTIONIQ DATA SUMMARY ===
Ball-by-ball records : 287,513
Total matches        : 1,209
Total seasons        : 19
Unique batters       : 728
Unique bowlers       : 569
Total runs scored    : 369,061
Total wickets        : 14,306
Date range           : 2008-04-18 → 2026-04-28

📁 Saved:
   data/processed/master_balls.csv
   data/processed/master_meta.csv

Next → 01_batting_analysis.ipynb
